# Wordle 游戏与 AgentLoop 设计

本节介绍 Wordle 游戏规则、G/Y/X 反馈机制，以及 verl 中 WordleAgentLoop 如何实现多轮交互。

---

## 1. Wordle 游戏规则

Wordle 是一个 6 轮猜词游戏：

1. 系统随机选择一个 5 字母秘密单词
2. 玩家每轮猜测一个 5 字母单词
3. 系统返回 G/Y/X 反馈：
   - **G（Green）**：字母正确且位置正确
   - **Y（Yellow）**：字母存在但位置错误
   - **X（Wrong）**：字母不在单词中
4. 最多 6 轮，猜中获胜

### 示例

```text
秘密单词: crane

第1轮 guess: apple  ->  Y X X X G  (a 位置错误，e 位置正确)
第2轮 guess: crane  ->  G G G G G  (全部猜中!)

第1轮 guess: house  ->  X X X X G  (e 位置正确)
第2轮 guess: crane  ->  G G G G G  (全部猜中!)
```

---

## 2. 多轮交互流程

在 RL 训练中，模型不是一次性给出答案，而是与游戏环境进行**多轮交互**：

<img src="./images/multi_turn.png" alt="Wordle 多轮交互流程" width="90%">

一次 rollout 不是单次模型输出，而是一段由 AgentLoop 管理的多轮对话。每一轮中，模型生成规范格式 `<guess>[word]</guess>`，环境解析猜词并返回 G/Y/X 反馈；如果没有猜中，反馈会作为下一轮上下文继续追加。

这类多轮交互让模型可以根据上一轮反馈调整下一轮猜测。训练时记录的是完整对话轨迹，因此后续还需要明确哪些 token 是模型生成的，哪些 token 是环境追加的。

---

## 3. WordleAgentLoop

WordleAgentLoop 是 verl AgentLoop 的自定义实现，负责管理上述多轮交互。它的核心职责：

1. **发送游戏 prompt**：初始的 system + game prompt
2. **接收模型猜词**：解析 `<guess>[word]</guess>` 标签
3. **计算 G/Y/X 反馈**：与秘密单词比对
4. **追加反馈到对话**：作为 user message
5. **重复直到猜中或达到最大轮次**

### Token 追踪机制

多轮交互中一个关键技术问题是 **token 追踪**——训练时需要区分哪些 token 是模型生成的（需要计算梯度），哪些是环境追加的（不计算梯度）。

<img src="./images/token_tracking.png" alt="Wordle AgentLoop token 追踪机制" width="90%">

ToolAgentLoop 会把多轮对话拼接到 `response_ids` 中，并用 `response_mask` 标记训练范围：`1` 表示 Assistant 生成的 token，需要参与 policy gradient；`0` 表示环境追加的 User feedback，只作为下一轮上下文，不计算梯度。

WordleAgentLoop 使用 ToolAgentLoop 模式实现 token 追踪：prompt_ids 单调增长，response_mask 标记哪些 token 需要计算梯度。这确保了训练数据的正确性。

---

## 4. 关键配置参数

| 参数 | 值 | 说明 |
|------|-----|------|
| `multi_turn.max_user_turns` | 6 | Wordle 最大猜词轮次 |
| `multi_turn.max_assistant_turns` | 6 | 模型最大回复轮次 |
| `multi_turn.format` | hermes | 多轮对话格式 |
| `rollout.agent.default_agent_loop` | wordle_agent | 使用 WordleAgentLoop |
| `rollout.agent.num_workers` | 4 | 并行 AgentLoop Worker 数 |

---

## 课后练习

1. （判断题）Wordle 游戏中，G（Green）表示字母正确且位置正确，Y（Yellow）表示字母存在但位置错误。

2. （判断题）WordleAgentLoop 的职责包括发送 prompt、接收猜词、计算反馈、追加反馈到对话。

3. （判断题）在多轮交互中，环境追加的 user feedback token 也需要计算梯度。

4. （单选题）WordleAgentLoop 使用 ToolAgentLoop 模式的 response_mask 中，值为 0 表示什么？
    A. 模型生成的 token（计算梯度）
    B. 环境追加的 token（不计算梯度）
    C. 需要删除的 token
    D. 无效的 token

5. （单选题）Wordle 训练中 `max_user_turns=6` 表示什么？
    A. 最多 6 个用户同时游戏
    B. Wordle 最大猜词轮次为 6
    C. 训练 batch 中有 6 个 prompt
    D. 每个 prompt 生成 6 条 rollout

In [ ]:
!cat ../cann-learning-hub/tutorials/rl_training_pipeline/03_wordle_rl_training/answer/03.02_answer.txt